# Scribble Evaluation Notebook
Load a scribble from a `.npy` file (Google Drive or local path), generate conditioned photos,
compute MMD vs target distribution, and check CLIP softmax gender scores.

## 1. Setup

In [ ]:
import os, sys, json, gc
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF
import torch
from pathlib import Path
from PIL import Image
from IPython.display import display
from huggingface_hub import login
from google.colab import userdata
import wandb



hf_token = userdata.get('HF')
if hf_token:
    login(token=hf_token)
else:
    login()

if 'google.colab' in str(get_ipython()):
    import getpass

    !pip install -q diffusers transformers accelerate xformers
    !pip install -q scikit-learn matplotlib Pillow

    github_token = userdata.get('GITHUB')
    if github_token:
        token = github_token
    else:
        token = getpass.getpass("Enter your GitHub personal access token: ")

    repo_url  = f"https://{token}@github.com/orineo1/conditional-matching-paper.git"
    repo_name = "conditional-matching-paper"
    branch    = "SD-add-vis-and-table"

    if not os.path.exists(repo_name):
        !git clone {repo_url}
    else:
        print(f"✅ Repo '{repo_name}' already cloned — pulling latest...")
        !cd {repo_name} && git pull

    !cd {repo_name} && git checkout {branch}

    repo_path = f"/content/{repo_name}"
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)

    print(f"\n✅ Repo ready. Branch: {branch}")
    print(f"📁 Python path: {repo_path}")

repo_path = f"/content/{repo_name}/SD_cond_SD_controlnet"
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

wandb_token = userdata.get('WANDB')
if wandb_token:
    wandb.login(key=wandb_token)
else:
    wandb.login()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

#Example

### 2. Load Scribble

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RUNS_ROOT        = "/content/drive/MyDrive/conditional-matching/runs"
CONTROLNET_SCALE = 0.5
N_EVAL           = 100
N_TARGETS        = 100
EVAL_PROMPT      = "a superrealistic professional photograph of"
MAN_PROMPT       = "a superrealistic portrait photograph of a man, studio lighting"
WOMAN_PROMPT     = "a superrealistic portrait photograph of a woman, studio lighting"
n_half           = N_TARGETS // 2

JOB_IDS = [
    44392389, 44388178, 44392384, 44392386, 44388185,
    44388183, 44392382, 44392395, 44388184, 44388179,
    44392394, 44392396, 44388181, 44392388, 44392392,
    44388177, 44392385, 44392390, 44392393, 44392391,
    44392383, 44392387, 44388182, 44388180, 44388176,
]


### 3. Load Models

In [ ]:
from models     import load_models
from clip_utils import load_clip_model

architect, sprinter    = load_models(device)
clip_model, clip_processor = load_clip_model(device)
print("✅ Models loaded.")


### 4. Helpers

In [ ]:

from generation    import generate_and_store_cs
from clip_utils    import encode_images_clip
from visualization import plot_row
from run_dps       import compute_clip_softmax
from metrics       import compute_mmd

def pil_to_tensor(pil_list):
    return torch.cat(
        [TF.to_tensor(img).unsqueeze(0) for img in pil_list], dim=0
    ).to(next(clip_model.parameters()).device)

def generate_eval_photos(scribble_pil, n=N_EVAL):
    sprinter.vae.to(dtype=torch.float16)
    photos = []
    with torch.no_grad():
        for start in range(0, n, 2):
            bs = min(2, n - start)
            result = sprinter(
                prompt=[EVAL_PROMPT] * bs,
                image=[scribble_pil] * bs,
                num_inference_steps=2, guidance_scale=0.0,
                controlnet_conditioning_scale=CONTROLNET_SCALE,
                output_type="pil",
            )
            photos.extend(result.images)
    sprinter.vae.to(dtype=torch.float32)
    return photos

def summarize(vals):
    a = np.array([float(v) for v in vals if v is not None and not np.isnan(float(v))])
    return (a.mean(), a.std()) if len(a) > 0 else (np.nan, np.nan)

def fmt(mean, std, decimals=3):
    fs = f"{{:.{decimals}f}}"
    return f"{fs.format(mean)} ± {fs.format(std)}"

## 5. Read All metrics.json and Build Table

In [ ]:
records = []
for jid in JOB_IDS:
    metrics_path = Path(RUNS_ROOT) / f"dps_main_{jid}" / "metrics.json"
    if not metrics_path.exists():
        print(f"  ⚠️  {jid}: metrics.json missing — skipping")
        continue
    with open(metrics_path) as f:
        m = json.load(f)

    lgd_g = m["lgd_cm_gender"]
    reg_g = m["regular_gender"]

    lgd_mmd = m["final_lgd_cm_mmd"]
    reg_mmd = m["final_regular_mmd"]

    records.append({
        "job_id":        jid,
        "lgd_mmd":       lgd_mmd,
        "reg_mmd":       reg_mmd,
        "mri":           (reg_mmd - lgd_mmd) / reg_mmd * 100,   # ← added
        "lgd_swd":       m["final_lgd_cm_swd"],
        "reg_swd":       m["final_regular_swd"],
        "lgd_pct_male":  lgd_g["n_male"] / (lgd_g["n_male"] + lgd_g["n_female"]) * 100,
        "reg_pct_male":  reg_g["n_male"] / (reg_g["n_male"] + reg_g["n_female"]) * 100,
        "lgd_conf_male":   lgd_g.get("mean_conf_male"),
        "lgd_conf_female": lgd_g.get("mean_conf_female"),
        "reg_conf_male":   reg_g.get("mean_conf_male"),
        "reg_conf_female": reg_g.get("mean_conf_female"),
        "opt_time_min":  m["optimization_time_sec"] / 60,
    })

print(f"Loaded {len(records)} runs.\n")

# ── Sort by MMD, but pick best by MRI ────────────────────────
records_sorted_mmd = sorted(records, key=lambda r: r["lgd_mmd"])
records_sorted_mri = sorted(records, key=lambda r: r["mri"], reverse=True)

top10_records = records_sorted_mmd[:10]
best          = records_sorted_mri[0]   # ← best by MRI

# ── Per-run table ─────────────────────────────────────────────
print(f"  {'Job ID':<12} {'LGD MMD':>8} {'Reg MMD':>8} {'MRI%':>7} {'%Male LGD':>10} {'%Male Reg':>10} {'Time(min)':>10}")
print("  " + "-" * 65)
for r in records_sorted_mri:   # show sorted by MRI
    top = "★" if r['job_id'] == best['job_id'] else " "
    print(f"{top} {r['job_id']:<12} {r['lgd_mmd']:>8.4f} {r['reg_mmd']:>8.4f} {r['mri']:>7.1f} {r['lgd_pct_male']:>10.1f} {r['reg_pct_male']:>10.1f} {r['opt_time_min']:>10.1f}")

print(f"\n★ Best run (MRI): {best['job_id']}  MRI={best['mri']:.1f}%  LGD MMD={best['lgd_mmd']:.4f}  Reg MMD={best['reg_mmd']:.4f}")
print(f"  Top-10 job IDs (by MMD): {[r['job_id'] for r in top10_records]}")

# ── Aggregate stats table ─────────────────────────────────────
import pandas as pd

def agg_stats(recs):
    lgd_mmd = [r["lgd_mmd"] for r in recs]
    reg_mmd = [r["reg_mmd"] for r in recs]
    mri     = [r["mri"]     for r in recs]
    return {
        "MMD — LGD-CM":          fmt(*summarize(lgd_mmd)),
        "MMD — Regular":         fmt(*summarize(reg_mmd)),
        "MRI (%)":               fmt(*summarize(mri), decimals=1),
        "SWD — LGD-CM":          fmt(*summarize([r["lgd_swd"]         for r in recs])),
        "SWD — Regular":         fmt(*summarize([r["reg_swd"]         for r in recs])),
        "% Male — LGD-CM":       fmt(*summarize([r["lgd_pct_male"]    for r in recs]), decimals=1),
        "% Male — Regular":      fmt(*summarize([r["reg_pct_male"]    for r in recs]), decimals=1),
        "Conf Male — LGD-CM":    fmt(*summarize([r["lgd_conf_male"]   for r in recs])),
        "Conf Female — LGD-CM":  fmt(*summarize([r["lgd_conf_female"] for r in recs])),
        "Conf Male — Regular":   fmt(*summarize([r["reg_conf_male"]   for r in recs])),
        "Conf Female — Regular": fmt(*summarize([r["reg_conf_female"] for r in recs])),
        "Opt. time (min)":       fmt(*summarize([r["opt_time_min"]    for r in recs]), decimals=1),
    }

df = pd.DataFrame({
    f"All ({len(records)})": agg_stats(records),
    "Top-10 (by MMD)":       agg_stats(top10_records),
})
display(df)

## 6. Load Best Run Images and Visualize

In [ ]:

best_jid = 44388183  #best['job_id']
run_dir  = Path(RUNS_ROOT) / f"dps_main_{best_jid}"


lgd_img    = Image.open(run_dir / "final_scribble_lgd_cm.png")
reg_img    = Image.open(run_dir / "final_scribble_regular.png")
source_img = Image.open(run_dir / "source_portrait.png")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, title in zip(
    axes,
    [source_img, lgd_img, reg_img],
    ['Source portrait', f'LGD-CM scribble\n(job {best_jid})', 'Regular scribble'],
):
    ax.imshow(img); ax.axis('off'); ax.set_title(title)
plt.suptitle(
    f"Best run: job {best_jid}  |  LGD-CM MMD={best['lgd_mmd']:.4f}  Regular MMD={best['reg_mmd']:.4f}",
    y=1.02
)
plt.tight_layout(); display(fig); plt.close()

# ── Use LGD-CM scribble as scribble_pil_1, regular as scribble_pil_2 ──
scribble_pil_1 = lgd_img
scribble_pil_2 = reg_img

## 7. Build Target Distribution


In [ ]:

print(f"Building target distribution ({N_TARGETS} images)...")
with torch.no_grad():
    man_images,   _ = generate_and_store_cs(sprinter, MAN_PROMPT,   scribble_pil_1, n_half, batch_size=2, cn_scale=CONTROLNET_SCALE)
    woman_images, _ = generate_and_store_cs(sprinter, WOMAN_PROMPT, scribble_pil_1, n_half, batch_size=2, cn_scale=CONTROLNET_SCALE)

with torch.no_grad():
    all_clip_embeddings = torch.cat([
        encode_images_clip(pil_to_tensor(man_images),   clip_model, clip_processor),
        encode_images_clip(pil_to_tensor(woman_images), clip_model, clip_processor),
    ], dim=0)

print(f"Target CLIP embeddings: {all_clip_embeddings.shape}")
plot_row(man_images,   f"Target Man ({n_half})",   count=n_half)
plot_row(woman_images, f"Target Woman ({n_half})", count=n_half)

### 5. Generate Eval Photos from Scribble

In [ ]:
print("Generating eval photos from LGD-CM scribble...")
eval_photos_1 = generate_eval_photos(scribble_pil_1)
print("Generating eval photos from Regular scribble...")
eval_photos_2 = generate_eval_photos(scribble_pil_2)

plot_row(eval_photos_1, "LGD-CM",  count=min(10, len(eval_photos_1)))
plot_row(eval_photos_2, "Regular", count=min(10, len(eval_photos_2)))

### 6. Compute MMD vs Target

In [ ]:

with torch.no_grad():
    eval_clip_1 = encode_images_clip(pil_to_tensor(eval_photos_1), clip_model, clip_processor)
    eval_clip_2 = encode_images_clip(pil_to_tensor(eval_photos_2), clip_model, clip_processor)

mmd_1 = compute_mmd(eval_clip_1, all_clip_embeddings).item()
mmd_2 = compute_mmd(eval_clip_2, all_clip_embeddings).item()
print(f"MMD LGD-CM  : {mmd_1:.6f}")
print(f"MMD Regular : {mmd_2:.6f}")
print(f"Delta       : {mmd_1 - mmd_2:.6f}")


### 7. CLIP Softmax Gender Scores

In [ ]:

results_1, eval_clip_np_1 = compute_clip_softmax(eval_photos_1, clip_model, clip_processor, MAN_PROMPT, WOMAN_PROMPT, device)
results_2, eval_clip_np_2 = compute_clip_softmax(eval_photos_2, clip_model, clip_processor, MAN_PROMPT, WOMAN_PROMPT, device)

for label, photos, results, mmd_val in [
    ("LGD-CM",  eval_photos_1, results_1, mmd_1),
    ("Regular", eval_photos_2, results_2, mmd_2),
]:
    n_male   = sum(1 for r in results if r["label"] == "male")
    n_female = sum(1 for r in results if r["label"] == "female")
    print(f"{label} — Male: {n_male}  Female: {n_female}  MMD: {mmd_val:.4f}")

    paired    = list(zip(photos, results))
    top_men   = sorted([(i, r) for i, r in paired if r["label"] == "male"],   key=lambda x: x[1]["p_male"],   reverse=True)[:5]
    top_women = sorted([(i, r) for i, r in paired if r["label"] == "female"], key=lambda x: x[1]["p_female"], reverse=True)[:5]

    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    for col, (img, r) in enumerate(top_men):
        axes[0, col].imshow(img)
        axes[0, col].set_title(f'p={r["p_male"]:.2f}', fontsize=9, color='royalblue')
        axes[0, col].axis('off')
    for col in range(len(top_men), 5):
        axes[0, col].axis('off')
    for col, (img, r) in enumerate(top_women):
        axes[1, col].imshow(img)
        axes[1, col].set_title(f'p={r["p_female"]:.2f}', fontsize=9, color='crimson')
        axes[1, col].axis('off')
    for col in range(len(top_women), 5):
        axes[1, col].axis('off')
    axes[0, 0].set_ylabel('Top 5 Men',   fontsize=11, color='royalblue', labelpad=8)
    axes[1, 0].set_ylabel('Top 5 Women', fontsize=11, color='crimson',   labelpad=8)
    fig.suptitle(f"{label} — {n_male}M / {n_female}F  |  MMD={mmd_val:.4f}", fontsize=13, fontweight='bold')
    plt.tight_layout(); display(fig); plt.close()


### 8. CLIP PCA — Eval vs Target

In [ ]:

from sklearn.decomposition import PCA
from matplotlib.patches import Patch

target_np = all_clip_embeddings.cpu().numpy()
combined  = np.vstack([target_np, eval_clip_np_1, eval_clip_np_2])
pca       = PCA(n_components=2)
coords    = pca.fit_transform(combined)

t_man   = coords[:n_half]
t_woman = coords[n_half:2*n_half]
e1_pts  = coords[2*n_half: 2*n_half + len(eval_clip_np_1)]
e2_pts  = coords[2*n_half + len(eval_clip_np_1):]

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(t_man[:,0],  t_man[:,1],  c='royalblue', s=60, alpha=0.7, label='Target (man)')
ax.scatter(t_woman[:,0],t_woman[:,1],c='crimson',   s=60, alpha=0.7, label='Target (woman)')
ax.scatter(e1_pts[:,0], e1_pts[:,1], c='darkorange',s=80, alpha=0.9, marker='x', label=f'LGD-CM (MMD={mmd_1:.4f})')
ax.scatter(e2_pts[:,0], e2_pts[:,1], c='limegreen', s=80, alpha=0.9, marker='+', label=f'Regular (MMD={mmd_2:.4f})')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', fontsize=11)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', fontsize=11)
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); display(fig); plt.close()

In [ ]:

def split_conf(results):
    return (
        [r["p_male"]   for r in results if r["label"] == "male"],
        [r["p_female"] for r in results if r["label"] == "female"],
    )

m1, f1 = split_conf(results_1)
m2, f2 = split_conf(results_2)

positions = [1, 1.8, 3.5, 4.3]
data      = [m1, m2, f1, f2]
colors    = ["darkorange", "steelblue", "darkorange", "steelblue"]

fig, ax = plt.subplots(figsize=(8, 5))
for pts, pos, col in zip(data, positions, colors):
    if pts:
        ax.boxplot(pts, positions=[pos], patch_artist=True, widths=0.5, showfliers=False,
                   medianprops=dict(color="black", linewidth=1.5),
                   boxprops=dict(facecolor=col, alpha=0.35, edgecolor="black"))
        jitter = np.random.uniform(-0.1, 0.1, size=len(pts))
        ax.scatter(np.full(len(pts), pos) + jitter, pts,
                   color=col, alpha=0.7, s=40, zorder=3, edgecolors="black", linewidth=0.5)

for pos, count, col in zip(positions, [len(m1), len(m2), len(f1), len(f2)], colors):
    ax.text(pos, -0.02, f"n={count}", transform=ax.get_xaxis_transform(),
            ha="center", va="top", color=col, fontsize=11)

ax.set_xticks([1.4, 3.9])
ax.set_xticklabels(["Male", "Female"], fontsize=13)
ax.tick_params(axis="x", pad=12)
ax.set_ylim(0.5, 1.02)
ax.set_ylabel("Confidence", fontsize=11)
ax.axvline(2.65, color="gray", lw=0.8, linestyle="--", alpha=0.3)
ax.grid(True, alpha=0.15, axis="y")
ax.legend(
    handles=[
        Patch(facecolor="darkorange", alpha=0.35, edgecolor="black", label="LGD-CM"),
        Patch(facecolor="steelblue",  alpha=0.35, edgecolor="black", label="Regular"),
    ],
    loc="upper center", bbox_to_anchor=(0.5, -0.07), ncol=2, frameon=True, fontsize=11,
)
plt.tight_layout(); display(fig); plt.close()

In [ ]:
# ## Re-evaluate All Runs with N=500 Samples

# from scipy import stats as scipy_stats

# N_EVAL_AGG = 500
# n_half_tgt = N_EVAL_AGG // 2  # 250 man + 250 woman for target

# # ── Build target distribution once ───────────────────────────
# print(f"Building target distribution ({N_EVAL_AGG} images)...")
# first_run_dir   = Path(RUNS_ROOT) / f"dps_main_{JOB_IDS[0]}"
# target_scribble = Image.open(first_run_dir / "final_scribble_regular.png")

# with torch.no_grad():
#     man_images_tgt,   _ = generate_and_store_cs(sprinter, MAN_PROMPT,   target_scribble, n_half_tgt, batch_size=2, cn_scale=CONTROLNET_SCALE)
#     woman_images_tgt, _ = generate_and_store_cs(sprinter, WOMAN_PROMPT, target_scribble, n_half_tgt, batch_size=2, cn_scale=CONTROLNET_SCALE)

# with torch.no_grad():
#     all_clip_tgt = torch.cat([
#         encode_images_clip(pil_to_tensor(man_images_tgt),   clip_model, clip_processor),
#         encode_images_clip(pil_to_tensor(woman_images_tgt), clip_model, clip_processor),
#     ], dim=0)
# print(f"Target CLIP embeddings: {all_clip_tgt.shape}")

# # ── Evaluate all runs ─────────────────────────────────────────
# eval_records = []

# for jid in JOB_IDS:
#     run_dir  = Path(RUNS_ROOT) / f"dps_main_{jid}"
#     lgd_path = run_dir / "final_scribble_lgd_cm.png"
#     reg_path = run_dir / "final_scribble_regular.png"

#     if not lgd_path.exists() or not reg_path.exists():
#         print(f"  ⚠️  {jid}: PNG missing — skipping")
#         continue

#     print(f"\n── Job {jid} ──")
#     lgd_scribble = Image.open(lgd_path)
#     reg_scribble = Image.open(reg_path)

#     for label, scribble in [("lgd_cm", lgd_scribble), ("regular", reg_scribble)]:

#         # ── Generate N_EVAL_AGG photos ────────────────────────
#         sprinter.vae.to(dtype=torch.float16)
#         photos = []
#         with torch.no_grad():
#             for start in range(0, N_EVAL_AGG, 2):
#                 bs = min(2, N_EVAL_AGG - start)
#                 result = sprinter(
#                     prompt=[EVAL_PROMPT] * bs,
#                     image=[scribble] * bs,
#                     num_inference_steps=2, guidance_scale=0.0,
#                     controlnet_conditioning_scale=CONTROLNET_SCALE,
#                     output_type="pil",
#                 )
#                 photos.extend(result.images)
#         sprinter.vae.to(dtype=torch.float32)

#         # ── CLIP embeddings + MMD ─────────────────────────────
#         with torch.no_grad():
#             clip_embs = encode_images_clip(
#                 pil_to_tensor(photos).to(next(clip_model.parameters()).device),
#                 clip_model, clip_processor,
#             )
#         mmd_val = compute_mmd(clip_embs, all_clip_tgt).item()

#         # ── Gender softmax ────────────────────────────────────
#         softmax_results, clip_np = compute_clip_softmax(
#             photos, clip_model, clip_processor, MAN_PROMPT, WOMAN_PROMPT, device
#         )
#         n_male   = sum(1 for r in softmax_results if r["label"] == "male")
#         n_female = sum(1 for r in softmax_results if r["label"] == "female")
#         pct_male = n_male / len(softmax_results) * 100

#         # ── Binomial p-value: H0=p(female)=0.5, H1=guided produces more female
#         binom_pval = scipy_stats.binomtest(n_female, n=len(softmax_results), p=0.5, alternative='greater').pvalue

#         eval_records.append({
#             "job_id":     jid,
#             "condition":  label,
#             "mmd":        mmd_val,
#             "n_male":     n_male,
#             "n_female":   n_female,
#             "pct_male":   pct_male,
#             "binom_pval": binom_pval,
#             "clip_np":    clip_np,
#             "softmax":    softmax_results,
#         })
#         print(f"  {label:8s}  MMD={mmd_val:.4f}  {n_male}M/{n_female}F  p={binom_pval:.4f}")

# print("\n✅ All runs evaluated.")

# # ── Build per-run summary ─────────────────────────────────────
# lgd_by_job = {r["job_id"]: r for r in eval_records if r["condition"] == "lgd_cm"}
# reg_by_job = {r["job_id"]: r for r in eval_records if r["condition"] == "regular"}

# eval_summary = []
# for jid in lgd_by_job:
#     if jid not in reg_by_job:
#         continue
#     l = lgd_by_job[jid]
#     r = reg_by_job[jid]
#     mri = (r["mmd"] - l["mmd"]) / r["mmd"] * 100
#     eval_summary.append({
#         "job_id":       jid,
#         "lgd_mmd":      l["mmd"],
#         "reg_mmd":      r["mmd"],
#         "mri":          mri,
#         "lgd_pct_male": l["pct_male"],
#         "reg_pct_male": r["pct_male"],
#         "binom_pval":   l["binom_pval"],
#     })

# eval_summary_mri = sorted(eval_summary, key=lambda r: r["mri"], reverse=True)
# top10_eval       = sorted(eval_summary, key=lambda r: r["lgd_mmd"])[:10]
# best_eval        = eval_summary_mri[0]

# # ── Per-run table ─────────────────────────────────────────────
# print(f"\n  {'Job ID':<12} {'LGD MMD':>8} {'Reg MMD':>8} {'MRI%':>7} {'%Male LGD':>10} {'%Male Reg':>10} {'p-val':>8}")
# print("  " + "-" * 70)
# for r in eval_summary_mri:
#     star = "★" if r["job_id"] == best_eval["job_id"] else " "
#     print(f"{star} {r['job_id']:<12} {r['lgd_mmd']:>8.4f} {r['reg_mmd']:>8.4f} {r['mri']:>7.1f} {r['lgd_pct_male']:>10.1f} {r['reg_pct_male']:>10.1f} {r['binom_pval']:>8.4f}")

# print(f"\n★ Best (MRI): {best_eval['job_id']}  MRI={best_eval['mri']:.1f}%  p={best_eval['binom_pval']:.4f}")
# print(f"  Top-10 job IDs (by MMD): {[r['job_id'] for r in top10_eval]}")

# # ── Aggregate stats table ─────────────────────────────────────
# import pandas as pd

# def agg_stats_eval(recs):
#     return {
#         "MMD — LGD-CM":   fmt(*summarize([r["lgd_mmd"]      for r in recs])),
#         "MMD — Regular":  fmt(*summarize([r["reg_mmd"]      for r in recs])),
#         "MRI (%)":        fmt(*summarize([r["mri"]          for r in recs]), decimals=1),
#         "% Male — LGD-CM":  fmt(*summarize([r["lgd_pct_male"] for r in recs]), decimals=1),
#         "% Male — Regular": fmt(*summarize([r["reg_pct_male"] for r in recs]), decimals=1),
#         "Binomial p-val": fmt(*summarize([r["binom_pval"]   for r in recs])),
#     }

# df_eval = pd.DataFrame({
#     f"All ({len(eval_summary)})": agg_stats_eval(eval_summary),
#     "Top-10 (by MMD)":            agg_stats_eval(top10_eval),
# })
# display(df_eval)